# FloraScan - Transfer Learning
Using pretrained MobileNetV2 for plant disease detection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow: {tf.__version__}")

## Configuration

In [ ]:
TRAIN_DIR = r'd:\Florascann\dataset_limited\train'
TEST_DIR = r'd:\Florascann\dataset_limited\test'

IMG_SIZE = 224  # MobileNetV2 default size
BATCH_SIZE = 32
EPOCHS = 15

NUM_CLASSES = len(os.listdir(TRAIN_DIR))
print(f"Classes: {NUM_CLASSES}")

## Load Data with Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, class_mode='categorical'
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

## Build Transfer Learning Model
Using pretrained MobileNetV2 (frozen) with custom classifier.

In [ ]:
# Load pretrained MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, 
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze base model
base_model.trainable = False

# Add custom classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print(f"Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")
print(f"Non-trainable parameters: {sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights]):,}")

## Train Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

print("Training Transfer Learning model...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=callbacks,
    verbose=1
)
print("Training completed!")

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], 'b-', label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
axes[0].set_title('Transfer Learning - Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], 'b-', label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], 'r-', label='Validation', linewidth=2)
axes[1].set_title('Transfer Learning - Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_generator, verbose=0)
train_acc = max(history.history['accuracy'])

print("="*50)
print("TRANSFER LEARNING RESULTS")
print("="*50)
print(f"Best Training Accuracy: {train_acc*100:.2f}%")
print(f"Best Validation Accuracy: {test_acc*100:.2f}%")
print(f"Overfitting Gap: {(train_acc - test_acc)*100:.2f}%")
print("="*50)
print("\n🚀 MAJOR IMPROVEMENT: Pretrained features work great!")
print("✅ MobileNetV2 already knows how to extract image features")
print("\n💡 NEXT: Fine-tune top layers for even better results")

In [ ]:
model.save(r'd:\Florascann\models\model_v3_transfer_learning.h5')
print("Model saved!")